# Retrieval-Augmented Generation (RAG) with PDFs

This project demonstrates a complete RAG pipeline using Python, LangChain, FAISS, and OpenAI API.

Workflow:
1. Load 2-3 PDF documents
2. Chunk the extracted text
3. Generate embeddings
4. Store embeddings in FAISS
5. Retrieve relevant chunks via semantic search
6. Generate a grounded answer from retrieved context

In [6]:
%pip install -U langchain langchain-openai langchain-community faiss-cpu pypdf tiktoken python-dotenv

## Dataset Setup

Add your own 2-3 PDF files into the `pdfs/` folder before running the notebook.

Example:
- pdfs/report_1.pdf
- pdfs/manual_2.pdf
- pdfs/policy_3.pdf

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
# Load Azure OpenAI settings from .env or environment
load_dotenv()

azure_api_key = os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT", "https://openai-api-management-gw.azure-api.net")
azure_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2023-05-15")
azure_embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002")
azure_chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o-mini")

if not azure_api_key:
    raise EnvironmentError("Missing API key. Set AZURE_OPENAI_API_KEY (or OPENAI_API_KEY) in .env or hardcode it in this cell.")

pdf_dir = Path("pdfs")
pdf_files = sorted(pdf_dir.glob("*.pdf"))

if len(pdf_files) < 2:
    raise ValueError("Please add at least 2 PDF files in the pdfs/ folder.")

print(f"Found {len(pdf_files)} PDF files:")
for p in pdf_files:
    print(f"- {p.name}")

print(f"Azure endpoint: {azure_endpoint}")
print(f"Embedding deployment: {azure_embedding_deployment}")
print(f"Chat deployment: {azure_chat_deployment}")

In [3]:
# Load pages from all PDFs
documents = []
for pdf_path in pdf_files:
    loader = PyPDFLoader(str(pdf_path))
    pages = loader.load()
    for page in pages:
        page.metadata["source_file"] = pdf_path.name
    documents.extend(pages)

print(f"Loaded {len(documents)} pages from {len(pdf_files)} files.")

In [4]:
# Chunk documents for retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

In [5]:
# Build embeddings and FAISS vector database
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version=azure_api_version,
    deployment=azure_embedding_deployment
)
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")

print("FAISS index built and saved to faiss_index/")

In [6]:
# Semantic retrieval
query = "What are the most important recommendations across these documents?"
k = 3
retrieved = vectorstore.similarity_search_with_score(query, k=k)

print("Top retrieved chunks:")
for i, (doc, score) in enumerate(retrieved, start=1):
    print(f"\nResult {i} | score={score:.4f}")
    print(f"Source: {doc.metadata.get('source_file', 'unknown')} | Page: {doc.metadata.get('page', 'n/a')}")
    print(doc.page_content[:400].replace("\n", " ") + "...")

In [7]:
# RAG answer generation
llm = AzureChatOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version=azure_api_version,
    deployment_name=azure_chat_deployment,
    temperature=0,
)
retrieved_docs = [doc for doc, _ in retrieved]
context = "\n\n".join(
    [
        f"Source: {d.metadata.get('source_file', 'unknown')} | Page: {d.metadata.get('page', 'n/a')}\n{d.page_content}"
        for d in retrieved_docs
    ]
)

prompt = f"""Use only the context below to answer the question.
If the answer is not in the context, say: Not found in the provided documents.

Question: {query}

Context:
{context}
"""

answer = llm.invoke(prompt).content
print("Answer:\n")
print(answer)

In [8]:
# Save sample output for LMS submission
output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
sample_output_path = output_dir / "sample_output.txt"

with open(sample_output_path, "w", encoding="utf-8") as f:
    f.write(f"Query: {query}\n\n")
    f.write("Top retrieved chunks:\n")
    for i, (doc, score) in enumerate(retrieved, start=1):
        f.write(f"\n[{i}] score={score:.4f}\n")
        f.write(
            f"source={doc.metadata.get('source_file', 'unknown')} page={doc.metadata.get('page', 'n/a')}\n"
        )
        f.write(doc.page_content[:600] + "\n")

    f.write("\nGenerated answer:\n")
    f.write(answer)

print(f"Sample output saved to: {sample_output_path.resolve()}")